In [ ]:
import json
from collections import OrderedDict

In [ ]:
with open("patient_features.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
print("Top-level type:", type(data))

In [ ]:
print("Number of records:", len(data))
first = data[2]

In [ ]:
if isinstance(first, dict):
    print("Field length summary (characters):")
    for k, v in first.items():
        length = len(json.dumps(v, ensure_ascii=False))
        print(f"- {k}: {length}")

In [ ]:
def m_preview(obj, max_chars=5000):
    features = obj.get("features")
    structural = {k: v for k, v in obj.items() if k != "features"}
    s = json.dumps(structural, indent=2, ensure_ascii=False)

    if len(s) > max_chars:
        s = s[:max_chars] + "\n... (truncated)"

    print(s)
    if isinstance(features, str):
        print('\n"features":')
        print("-" * 60)
        for i in range(0, len(features), 50):
            print(features[i:i+50])
        print("-" * 60)

print("Record preview:\n")
m_preview(first)

In [ ]:
import json
import pandas as pd
import statistics
from collections import Counter

def iter_json_array(fp, buf_size=1 << 20):
    dec = json.JSONDecoder()
    ch = fp.read(1)
    while ch and ch.isspace():
        ch = fp.read(1)
    if ch != "[":
        raise ValueError("Not a JSON array")
    buf = ""
    idx = 0
    while True:
        if idx >= len(buf):
            more = fp.read(buf_size)
            if not more:
                return
            buf = more
            idx = 0

        while True:
            while idx < len(buf) and buf[idx].isspace():
                idx += 1
            if idx < len(buf):
                break
            more = fp.read(buf_size)
            if not more:
                return
            buf = buf[idx:] + more
            idx = 0

        if buf[idx] == "]":
            return
        if buf[idx] == ",":
            idx += 1
            continue

        try:
            obj, end = dec.raw_decode(buf, idx)
        except ValueError:
            more = fp.read(buf_size)
            if not more:
                raise
            buf = buf[idx:] + more
            idx = 0
            continue

        idx = end
        yield obj

        if idx > (1 << 20):
            buf = buf[idx:]
            idx = 0

def iter_records(path):
    with open(path, "r", encoding="utf-8") as f:
        first = ""
        while True:
            ch = f.read(1)
            if not ch:
                return
            if not ch.isspace():
                first = ch
                break
        f.seek(0)
        if first == "[":
            for rec in iter_json_array(f):
                yield rec
        else:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                yield json.loads(line)

path = "patient_features.json"

key_counter = Counter()
note_type_counter = Counter()
csn_counts = Counter()

csns = []
n_notes_list = []
n_meds_list = []
notes_chars_total_list = []
note_chars_max_list = []
note_chars_median_list = []
med_chars_total_list = []
med_chars_max_list = []

bad_records = 0
bad_features = 0

for rec in iter_records(path):
    try:
        if isinstance(rec, dict):
            key_counter.update(rec.keys())
        feats = rec.get("features") if isinstance(rec, dict) else None
        if isinstance(feats, str):
            b = json.loads(feats)
        elif isinstance(feats, dict):
            b = feats
        else:
            bad_features += 1
            continue

        demo0 = (b.get("demographics") or [{}])[0]
        csn = demo0.get("csn")
        csn_counts[csn] += 1

        notes = b.get("binary") or []
        meds = b.get("medication_orders") or []

        note_lens = []
        for n in notes:
            note_lens.append(len(n.get("note", "") or ""))
            note_type_counter[n.get("note_type", "<missing>")] += 1

        med_lens = [len(m.get("dosage_text", "") or "") for m in meds]

        csns.append(csn)
        n_notes_list.append(len(notes))
        n_meds_list.append(len(meds))
        notes_chars_total_list.append(sum(note_lens))
        note_chars_max_list.append(max(note_lens) if note_lens else 0)
        note_chars_median_list.append(statistics.median(note_lens) if note_lens else 0)
        med_chars_total_list.append(sum(med_lens))
        med_chars_max_list.append(max(med_lens) if med_lens else 0)

    except Exception:
        bad_records += 1

stat = pd.DataFrame({
    "csn": csns,
    "n_notes": n_notes_list,
    "n_meds": n_meds_list,
    "notes_chars_total": notes_chars_total_list,
    "note_chars_max": note_chars_max_list,
    "note_chars_median": note_chars_median_list,
    "med_chars_total": med_chars_total_list,
    "med_chars_max": med_chars_max_list,
})

print("Parsed cases:", len(stat))
print("Bad records:", bad_records)
print("Bad features:", bad_features)

print("\nTop-level record keys (count across cases):")
for k, v in key_counter.most_common():
    print(f"  {k}: {v}")

print("\nUnique CSN:", stat["csn"].nunique(), "/", len(stat))
print("Duplicated CSN rows:", sum(1 for _, c in csn_counts.items() if c > 1))

print("\nCase-level summary:")
print(stat.drop(columns=["csn"]).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T)

print("\nTop note_type:")
for t, c in note_type_counter.most_common(30):
    print(f"  {t}: {c}")
